# BSL Word Transformer — GPU training on Google Colab

Trains the full 12-run grid ({transformer, lstm} x {aug, noaug} x seeds
{42, 43, 44}) on a Colab GPU, then copies results back to Google Drive.

Everything needed for training ships with the repository: cached landmark
arrays (`data/landmarks/`), `data/metadata.csv`, `data/vocabulary.csv` and
the frozen `data/splits.json`. No Drive upload is required.

**Before running: Runtime > Change runtime type > T4 GPU.**

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print('torch sees CUDA:', torch.cuda.is_available())

## 2. Clone the repository

In [ ]:
REPO_URL = 'https://github.com/froggy238/bsl-word-transformer.git'

!git clone {REPO_URL}
%cd bsl-word-transformer

## 3. Install the few packages Colab lacks

Deliberately *not* `pip install -r requirements.txt` here: Colab already
ships a CUDA-enabled torch, numpy, pandas, scikit-learn and matplotlib, and
replacing its torch with the CPU-pinned local version would slow training or
break CUDA. Training needs no MediaPipe (landmarks are precomputed). The
exact package versions used by every run are recorded in that run's
`env.txt`, which preserves reproducibility.

In [ ]:
!pip install -q pyyaml tqdm

## 4. Verify the data shipped with the clone

In [ ]:
import json
from pathlib import Path

n = sum(1 for _ in Path('data/landmarks').rglob('*.npz'))
splits = json.load(open('data/splits.json'))
print(f'{n} cached landmark files')
print(f"{len(splits['train'])} train / {len(splits['val'])} val clips, "
      f"{len(splits['label_list'])} classes")
assert n >= 290 and len(splits['label_list']) == 50

## 5. Train the 12-run grid

`check=True` aborts the loop on the first failing run so problems are not
silently skipped. Each run writes `results/runs/<run_id>/` (config copy,
env.txt, metrics.csv, best.pt). About 12 x 2-4 min on a T4.

In [ ]:
import glob
import subprocess
import sys
from pathlib import Path

configs = sorted(glob.glob('configs/*.yaml'))
print(f'{len(configs)} configs:', [Path(c).stem for c in configs])

for cfg in configs:
    print(f'\n===== {cfg} =====')
    subprocess.run(
        [sys.executable, '-m', 'src.train', '--config', cfg,
         '--device', 'cuda'],
        check=True)

## 6. Summary: best validation accuracy per run

In [ ]:
import numpy as np
import torch
from pathlib import Path

accs = {}
for run_dir in sorted(Path('results/runs').iterdir()):
    ckpt = run_dir / 'best.pt'
    if ckpt.is_file():
        c = torch.load(ckpt, map_location='cpu')
        accs[run_dir.name] = c['val_acc']
        print(f"{run_dir.name:<26} best val acc {c['val_acc']:.4f} "
              f"(epoch {c['epoch']})")

print()
for group in ('transformer_aug', 'transformer_noaug', 'lstm_aug', 'lstm_noaug'):
    vals = [v for k, v in accs.items() if k.startswith(group + '_s')]
    if vals:
        print(f'{group:<20} {np.mean(vals):.3f} \u00b1 {np.std(vals):.3f} (n={len(vals)})')

## 7. Save results to Google Drive

Colab VMs are ephemeral — run this before the runtime disconnects, or the
checkpoints and metrics are lost.

In [ ]:
import shutil
from google.colab import drive

drive.mount('/content/drive')
out = shutil.make_archive('/content/drive/MyDrive/bsl-results', 'zip', 'results')
print('results archived to', out)